Se extrae y limpian canciones para crear archivo "ITEMS.CSV"

# Importaciones

In [9]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

# Ruta flexible para poder ejecutar el notebook desde distintas carpetas.
RAW_PATH = Path('data/raw/dataset.csv')
if not RAW_PATH.exists():
	RAW_PATH = Path('../../data/raw/dataset.csv')
 
OUTPUT_PATH = Path('/data/processed/items.csv')
if not OUTPUT_PATH.exists():
	OUTPUT_PATH = Path('../../data/processed/items.csv')

# Carga de Dataset Original

In [10]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Shape original: {df_raw.shape}')
df_raw.head(3)

Shape original: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


# Normalización

In [11]:
# Verificar rango de features de audio (deben estar entre 0 y 1, excepto loudness y tempo)

df = df_raw.copy()

audio_features = ['danceability', 'energy', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence']

for feat in audio_features:
    fuera = df[(df[feat] < 0) | (df[feat] > 1)][feat].count()
    if fuera > 0:
        print(f'{feat}: {fuera} valores fuera del rango [0, 1]')
    else:
        print(f'{feat}: OK')

danceability: OK
energy: OK
speechiness: OK
acousticness: OK
instrumentalness: OK
liveness: OK
valence: OK


In [12]:
#Normalizar loudness y tempo al rango [0, 1] para que sean comparables
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[['loudness_norm', 'tempo_norm']] = scaler.fit_transform(df[['loudness', 'tempo']])

print('loudness_norm:', df['loudness_norm'].min(), '->', df['loudness_norm'].max())
print('tempo_norm:   ', df['tempo_norm'].min(),    '->', df['tempo_norm'].max())

loudness_norm: 0.0 -> 0.9999999999999999
tempo_norm:    0.0 -> 1.0


# Renombramiento de Columnas

In [13]:
# Normalizar loudness y tempo antes de seleccionar
df['loudness_norm'] = (df['loudness'] - df['loudness'].min()) / (df['loudness'].max() - df['loudness'].min())
df['tempo_norm'] = (df['tempo'] - df['tempo'].min()) / (df['tempo'].max() - df['tempo'].min())

# Columnas que nos interesan
# ITEM_ID es obligatorio para Amazon Personalize

items = df[[
    'track_id',           # ITEM_ID
    'track_name',         # titulo
    'artists',            # artista
    'album_name',         # album
    'track_genre',        # genero
    'popularity',         # popularidad
    'duration_ms',        # duracion_ms
    'explicit',           # explicito
    # Features de audio para Content-Based Filtering
    'danceability',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'loudness_norm',
    'tempo_norm'
]].copy()

# Renombrar para consistencia con el esquema del proyecto
items = items.rename(columns={
    'track_id'    : 'ITEM_ID',
    'track_name'  : 'titulo',
    'artists'     : 'artista',
    'album_name'  : 'album',
    'track_genre' : 'genero',
    'popularity'  : 'popularidad',
    'duration_ms' : 'duracion_ms',
    'explicit'    : 'explicito'
})

print(f'Shape final de items: {items.shape}')
items.head()

Shape final de items: (114000, 17)


,ITEM_ID,titulo,artista,album,genero,popularidad,duracion_ms,explicito,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,loudness_norm,tempo_norm
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,230666,False,0.676,0.4610,0.1430,0.0322,0.000001,0.3580,0.715,0.791392,0.361245
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,149610,False,0.420,0.1660,0.0763,0.9240,0.000006,0.1010,0.267,0.597377,0.318397
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,210826,False,0.438,0.3590,0.0557,0.2100,0.000000,0.1170,0.120,0.736123,0.313643
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,201933,False,0.266,0.0596,0.0363,0.9050,0.000071,0.1320,0.143,0.573701,0.746758
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,198853,False,0.618,0.4430,0.0526,0.4690,0.000000,0.0829,0.167,0.737103,0.492863


# Verificación

In [14]:
# ITEM_ID únicos
print(f'\nITEM_IDs únicos: {items["ITEM_ID"].nunique()} / {len(items)}')

# Géneros finales
print(f'\nGéneros disponibles ({items["genero"].nunique()}):'
      f'\n{sorted(items["genero"].unique())}')


ITEM_IDs únicos: 89741 / 114000

Géneros disponibles (114):
['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime', 'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british', 'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club', 'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house', 'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk', 'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie', 'indie-pop', 'industrial', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'jazz', 'k-pop', 'kids', 'latin', 'latino', 'malay', 'mandopop', 'metal', 'metalcore', 'minimal-techno', 'mpb', 'new-age', 'opera', 'pagode', 'party', 'piano', 'pop', 'pop-film', 'power-pop', 'progressive-house', 'psych-rock',

In [15]:
# Muestra de los datos finales
items.sample(5, random_state=42)

,ITEM_ID,titulo,artista,album,genero,popularidad,duracion_ms,explicito,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence,loudness_norm,tempo_norm
113186,6KwkVtXm8OUp2XffN5k7lY,No Other Name,Hillsong Worship,No Other Name,world-music,50,440247,False,0.369,0.598,0.0304,0.00511,0.000000,0.176,0.0466,0.786989,0.608180
42819,2dp5I5MJ8bQQHDoFaNRFtX,Failed Organum,Internal Rot,Grieving Birth,grindcore,11,93933,False,0.171,0.997,0.1180,0.00521,0.801000,0.420,0.0294,0.849842,0.502206
59311,5avw06usmFkFrPjX8NxC40,"Save the Trees, Pt. 1",Zhoobin Askarieh;Ali Sasha,Noise A Noise 20.4-1,iranian,0,213578,False,0.173,0.803,0.1440,0.61300,0.001910,0.195,0.0887,0.729889,0.310488
91368,75hT0hvlESnDJstem0JgyR,Merry Christmas,Bryan Adams,All I Want For Christmas Is You,rock,0,151387,False,0.683,0.511,0.0279,0.40600,0.000197,0.111,0.5980,0.812626,0.451946
61000,4bY2oZGA5Br3pTE1Jd1IfY,月の大きさ,Nogizaka46,バレッタ TypeD,j-idol,57,236293,False,0.555,0.941,0.0481,0.48400,0.000000,0.266,0.8130,0.855243,0.380023


# Exportación de CSV

In [16]:
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

items.to_csv(OUTPUT_PATH, index=False)
print(f'items.csv guardado en: {OUTPUT_PATH}')
print(f'{len(items):,} canciones exportadas')
print(f'{items.shape[1]} columnas: {list(items.columns)}')

items.csv guardado en: ..\..\data\processed\items.csv
114,000 canciones exportadas
17 columnas: ['ITEM_ID', 'titulo', 'artista', 'album', 'genero', 'popularidad', 'duracion_ms', 'explicito', 'danceability', 'energy', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'loudness_norm', 'tempo_norm']
